In [1]:
import sys
import os
sys.path.append("..")

from src.resampling import resamplingImage 
from src.metadata import extraer_info_tecnica

from tqdm import tqdm
import SimpleITK as sitk 
import pandas as pd
import numpy

In [2]:
df = pd.read_pickle('../data/metadata/metadata.pkl')

In [3]:
df.iloc

In [6]:
paths = df['Path'].tolist()
output_dir = "../data/processed/resampled"
os.makedirs(output_dir, exist_ok=True)
target_spacing = (0.4, 0.4, 0.6)

for path in tqdm(paths):
    try:
        base = os.path.splitext(os.path.basename(path))[0]
        output_name = base + ".nii.gz"
        output_path = os.path.join(output_dir, output_name)

        if os.path.exists(output_path):
            continue

        img = resamplingImage(path, target_spacing)
        sitk.WriteImage(img, output_path)

    except Exception as e:
        print("Error:", path)
        print(e)
    
    

100%|██████████| 1040/1040 [25:22<00:00,  1.46s/it]


In [9]:
path='/media/mrsmile/IA/tesis/data/processed/resampled/228.img.nii.gz'
idImage, size, spacing, dimension =extraer_info_tecnica(path)
print(idImage)
print(size)
print(spacing)

228.img.nii.gz
(408, 408, 172)
(0.4000000059604645, 0.4000000059604645, 0.6000000238418579)


In [4]:
import os
import SimpleITK as sitk
import pandas as pd
from tqdm import tqdm

def extraer_info_tecnica(path):
    img       = sitk.ReadImage(path)
    size      = img.GetSize()
    spacing   = img.GetSpacing()
    dimension = img.GetDimension()
    idImage   = os.path.splitext(os.path.splitext(os.path.basename(path))[0])[0]
    return idImage, size, spacing, dimension

RESAMPLED_DIR = '/media/mrsmile/IA/tesis/data/processed/resampled'
CSV_OUT       = '/media/mrsmile/IA/tesis/data/metadata/resampled_sizes.csv'

files = [f for f in os.listdir(RESAMPLED_DIR) if f.endswith('.nii.gz')]
print(f'Archivos encontrados: {len(files)}')

records = []
errors  = []

for fname in tqdm(files):
    path = os.path.join(RESAMPLED_DIR, fname)
    try:
        idImage, size, spacing, dimension = extraer_info_tecnica(path)
        records.append({
            'file':      fname,
            'id':        idImage,
            'dim_x':     size[0],
            'dim_y':     size[1],
            'dim_z':     size[2],
            'sp_x':      round(spacing[0], 4),
            'sp_y':      round(spacing[1], 4),
            'sp_z':      round(spacing[2], 4),
            'dimension': dimension,
        })
    except Exception as e:
        errors.append({'file': fname, 'error': str(e)})

df = pd.DataFrame(records)

# ── Exportar CSV
os.makedirs(os.path.dirname(CSV_OUT), exist_ok=True)
df.to_csv(CSV_OUT, index=False)
print(f'\nCSV guardado: {CSV_OUT}')
print(f'Total procesados: {len(df)}')
print(f'Errores:          {len(errors)}')

if errors:
    print('\nArchivos con error:')
    for e in errors:
        print(f"  {e['file']}: {e['error']}")

# ── Resumen
print(f'\n── Tamaños únicos (dim_x, dim_y, dim_z):')
print(df.groupby(['dim_x','dim_y','dim_z']).size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .to_string(index=False))

print(f'\n── Spacings únicos:')
print(df.groupby(['sp_x','sp_y','sp_z']).size()
        .reset_index(name='count')
        .to_string(index=False))

print(f'\n── Estadísticas dim_z (variable por FOV):')
print(df['dim_z'].describe())

df.head()

Archivos encontrados: 1040


100%|██████████| 1040/1040 [03:20<00:00,  5.18it/s]


CSV guardado: /media/mrsmile/IA/tesis/data/metadata/resampled_sizes.csv
Total procesados: 1040
Errores:          0

── Tamaños únicos (dim_x, dim_y, dim_z):
 dim_x  dim_y  dim_z  count
   408    408    229     96
   408    408    172     76
   462    462    229     26
   442    442    229     21
   448    448    229     21
   470    470    229     19
   450    450    229     18
   480    480    229     17
   445    445    229     17
   435    435    229     17
   478    478    229     16
   440    440    229     16
   455    455    229     16
   460    460    229     16
   472    472    229     15
   490    490    229     15
   438    438    229     15
   468    468    229     14
   458    458    229     13
   482    482    229     13
   465    465    229     12
   485    485    229     12
   452    452    229     12
   430    430    229     11
   422    422    229     10
   488    488    229     10
   500    500    229      9
   495    495    229      9
   432    432    229      8
  

,file,id,dim_x,dim_y,dim_z,sp_x,sp_y,sp_z,dimension
0,855.img.nii.gz,855.img,485,485,229,0.4,0.4,0.6,3
1,669.img.nii.gz,669.img,500,500,191,0.4,0.4,0.6,3
2,62.img.nii.gz,62.img,408,408,188,0.4,0.4,0.6,3
3,694.img.nii.gz,694.img,408,408,181,0.4,0.4,0.6,3
4,109.img.nii.gz,109.img,460,460,229,0.4,0.4,0.6,3


In [5]:
# ── Verificación de tamaños uniformes
print('═' * 50)
print('VERIFICACIÓN DE UNIFORMIDAD')
print('═' * 50)

# Tamaño más común
most_common = df.groupby(['dim_x','dim_y','dim_z']).size().idxmax()
print(f'\nTamaño más frecuente: {most_common}')

# Imágenes que NO coinciden con el tamaño más común
df_diff = df[
    (df['dim_x'] != most_common[0]) |
    (df['dim_y'] != most_common[1]) |
    (df['dim_z'] != most_common[2])
].copy()

if len(df_diff) == 0:
    print('\n✅ TODAS las imágenes tienen el mismo tamaño')
else:
    print(f'\n  {len(df_diff)} imágenes tienen tamaño DIFERENTE:')
    print(df_diff[['file','dim_x','dim_y','dim_z','sp_x','sp_y','sp_z']]
          .to_string(index=False))

# ── Verificación de spacing uniforme
most_common_sp = df.groupby(['sp_x','sp_y','sp_z']).size().idxmax()
df_diff_sp = df[
    (df['sp_x'] != most_common_sp[0]) |
    (df['sp_y'] != most_common_sp[1]) |
    (df['sp_z'] != most_common_sp[2])
].copy()

print(f'\nSpacing más frecuente: {most_common_sp}')
if len(df_diff_sp) == 0:
    print(' TODAS las imágenes tienen el mismo spacing')
else:
    print(f'  {len(df_diff_sp)} imágenes tienen spacing DIFERENTE:')
    print(df_diff_sp[['file','dim_x','dim_y','dim_z','sp_x','sp_y','sp_z']]
          .to_string(index=False))

# ── Resumen final
print('\n── Resumen general:')
print(f'   Total imágenes:        {len(df)}')
print(f'   Tamaños únicos:        {df.groupby(["dim_x","dim_y","dim_z"]).ngroups}')
print(f'   Spacings únicos:       {df.groupby(["sp_x","sp_y","sp_z"]).ngroups}')
print(f'   dim_z min/max:         {df["dim_z"].min()} / {df["dim_z"].max()}')
print('\n── Nota: dim_z variable es ESPERADO')
print('   (depende del FOV original de cada caso)')
print('   dim_x y dim_y deben ser iguales en todos')

══════════════════════════════════════════════════
VERIFICACIÓN DE UNIFORMIDAD
══════════════════════════════════════════════════

Tamaño más frecuente: (np.int64(408), np.int64(408), np.int64(229))

  944 imágenes tienen tamaño DIFERENTE:
              file  dim_x  dim_y  dim_z  sp_x  sp_y  sp_z
    855.img.nii.gz    485    485    229   0.4   0.4   0.6
    669.img.nii.gz    500    500    191   0.4   0.4   0.6
     62.img.nii.gz    408    408    188   0.4   0.4   0.6
    694.img.nii.gz    408    408    181   0.4   0.4   0.6
    109.img.nii.gz    460    460    229   0.4   0.4   0.6
    508.img.nii.gz    470    470    229   0.4   0.4   0.6
    622.img.nii.gz    410    410    229   0.4   0.4   0.6
Diseased_19.nii.gz    553    553    175   0.4   0.4   0.6
    659.img.nii.gz    468    468    186   0.4   0.4   0.6
    925.img.nii.gz    472    472    229   0.4   0.4   0.6
    368.img.nii.gz    442    442    229   0.4   0.4   0.6
    615.img.nii.gz    432    432    201   0.4   0.4   0.6
    51